# AI-Based Retail Demand Forecasting & Inventory Optimization
## Academic Project: Exploratory Data Analysis & Model Exploration

This notebook provides a complete walk-through of the data ingestion, feature engineering, and model training workflow using the **Corporación Favorita** store sales dataset.

In [ ]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import config
from src.data_pipeline import load_processed
from src.features import build_features
from src.train_model import load_models, train_quantile_models
from src.inventory import InventoryInputs, compute_inventory
from src.explainability import global_feature_importance

sns.set_theme(style="whitegrid")
print("Setup complete!")

### 1. Load Preprocessed Retail Sales Data

In [ ]:
df = load_processed()
print(f"Total records: {len(df):,}")
print(f"Stores available: {df['store_nbr'].unique().tolist()}")
print(f"Product families: {df['family'].unique().tolist()}")
df.head()

### 2. Exploratory Sales Trends

In [ ]:
# Plot total sales by product family
family_totals = df.groupby('family')['sales'].sum().sort_values(ascending=False)

plt.figure(figsize=(10, 5))
sns.barplot(x=family_totals.values, y=family_totals.index, palette="Blues_r")
plt.title("Total Sales Volume by Product Family", fontsize=14)
plt.xlabel("Units Sold")
plt.ylabel("Family")
plt.tight_layout()
plt.show()

### 3. Leakage-Free Feature Engineering Inspection

In [ ]:
feat_df = build_features(df, is_training=True)
print(f"Feature matrix shape: {feat_df.shape}")

# Check lag and rolling features
cols_to_check = ['date', 'store_nbr', 'family', 'sales', 'lag_1', 'lag_7', 'roll_mean_7']
feat_df[cols_to_check].dropna().head(10)

### 4. Quantile Model Inference & Forecast Verification

In [ ]:
q10_model, q50_model, q90_model, meta = load_models()
print("Validation metrics on held-out test data:")
for k, v in meta['metrics'].items():
    print(f"  {k}: {round(v, 4) if isinstance(v, float) else v}")

### 5. Inventory Optimization Scenario

In [ ]:
from src.forecasting import generate_forecast

# Generate 10-day forecast for Store 3, BEVERAGES
fc_df = generate_forecast(
    history_df=df,
    q10_model=q10_model,
    q50_model=q50_model,
    q90_model=q90_model,
    meta=meta,
    store_nbr=3,
    family="BEVERAGES",
    horizon=10
)

inputs = InventoryInputs(
    on_hand=500.0,
    on_order=100.0,
    backorders=0.0,
    review_period=7,
    lead_time=3,
    service_level=0.95
)

res = compute_inventory(inputs, fc_df)
print(f"Net Inventory Position: {res.net_inventory_position} units")
print(f"Expected 10-day Demand: {res.median_demand_H} units")
print(f"Safety Stock:           {res.safety_stock} units")
print(f"Recommended Order:      {res.order_quantity} units")
print("\n--- Calculation Walkthrough ---")
print(res.explanation)

### 6. Global Feature Importance (SHAP)

In [ ]:
imp = global_feature_importance(q50_model, meta['feature_cols'], top_n=10)
plt.figure(figsize=(8, 4))
sns.barplot(data=imp, x='importance', y='feature', palette="viridis")
plt.title("LightGBM Median Model Feature Importance")
plt.xlabel("Mean |SHAP| Importance")
plt.ylabel("Feature")
plt.tight_layout()
plt.show()